# Module 06: Decorators and Context Managers

## Interview Pro-Tip

"Are Python decorators like Java annotations?" — This is a TRAP question.
The answer is NO, they are fundamentally different. Understanding WHY is
what separates candidates who get it from those who don't.

## Java Recall

```java
// Annotations are passive metadata
@Timed  // This attaches metadata to the method
@Retry(maxAttempts = 3)
public String fetchData() {
    // Some framework reads the annotation and applies behavior
    // The annotation itself does NOTHING at runtime
}

// try-with-resources for resource management
try (FileWriter f = new FileWriter("output.txt")) {
    f.write("data");
} // Auto-close guaranteed
```

Java annotations are **metadata** — they don't change the code. A framework
(Spring, JUnit, AspectJ) reads them and acts on them. The annotation exists
without the framework — it's just inert information.

## Python Way

```python
# Decorators are ACTIVE transformations at definition time
@timer  # This REPLACES the function with a wrapped version
def slow_function():
    time.sleep(1)
    return "done"

# @timer is syntactic sugar for:
# slow_function = timer(slow_function)

# Context manager protocol
with open("file.txt", "w") as f:  # __enter__ called
    f.write("data")
# __exit__ called automatically — even if an exception occurs
```

**Critical difference:** `@timer` transforms the function **at definition time**.
The original function is gone — replaced by the wrapped version. No framework needed.
This is a language feature, not a configuration system.

## Key Differences

| Concept | Java | Python |
|---------|------|--------|
| Annotation / Decorator | Metadata — inert without framework | Function transformation at definition time |
| Activation | Framework/container processes it | Language runtime applies it |
| Without framework | Annotation exists, does nothing | Decorator still works — no framework needed |
| Arguments | `@Retry(max=3)` — attribute syntax | `@retry(max_attempts=3)` — function call that returns decorator |
| Resource management | try-with-resources (AutoCloseable) | `with` statement (context manager protocol) |
| Protocol for `with` | `AutoCloseable.close()` | `__enter__()` + `__exit__()` |

## Deep Dive: How Decorators Work

```python
# A decorator is just a function that takes a function and returns a function
def my_decorator(func):
    def wrapper(*args, **kwargs):
        print("Before")
        result = func(*args, **kwargs)
        print("After")
        return result
    return wrapper

# These are equivalent:
@my_decorator
def say_hello(): ...

def say_hello(): ...
say_hello = my_decorator(say_hello)  # Same thing!
```

**Parameterized decorators** add one more level:
```python
def retry(max_attempts):        # Level 1: takes config
    def decorator(func):         # Level 2: takes the function
        def wrapper(*args):      # Level 3: replaces the function
            for _ in range(max_attempts):
                try: return func(*args)
                except: pass
            raise
        return wrapper
    return decorator

@retry(max_attempts=3)  # retry(3) returns the decorator
def flaky(): ...
```

## Your Practice

Open `initial/practice.py`. Implement:

1. **@timer** — Simple decorator (4-6 lines)
2. **@retry(max_attempts, exceptions)** — Parameterized decorator (10-12 lines)
3. **@memoize** — Caching decorator (8-10 lines)
4. **TimedOpen** — Context manager class (15-20 lines)
5. **@validate_types** — Runtime type checking decorator (15-18 lines)

Run: `python run_test.py initial` / `python run_test.py complete`.

## References

- [PEP 318 — Decorators for Functions and Methods](https://peps.python.org/pep-0318/)
- [PEP 343 — The "with" Statement](https://peps.python.org/pep-0343/)
- [functools.wraps](https://docs.python.org/3/library/functools.html#functools.wraps)
